# RAG Application Example - Code Analysis

This notebook demonstrates a real-world RAG application for analyzing code files.
The example shows how to query code repositories and get intelligent responses about code patterns, best practices, and potential issues.

In [27]:
from dotenv import load_dotenv

load_dotenv(dotenv_path='.env')

True

In [28]:
from llama_index.llms.nvidia import NVIDIA
from llama_index.core import Settings

# Initialize with an NVIDIA model (e.g., Llama 3)
llm = NVIDIA(model="openai/gpt-oss-120b")
Settings.llm = llm

print(f"Using LLM: {llm.model}")

Using LLM: openai/gpt-oss-120b


In [29]:
from llama_index.embeddings.nvidia import NVIDIAEmbedding
from llama_index.core import Settings

embedder = NVIDIAEmbedding(model="nvidia/nv-embedqa-e5-v5")
Settings.embed_model = embedder

print(f"Using Embedding model: {embedder.model}")

Using Embedding model: nvidia/nv-embedqa-e5-v5


In [30]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.llms.nvidia import NVIDIA
from llama_index.embeddings.nvidia import NVIDIAEmbedding
from llama_index.core.text_splitter import TokenTextSplitter

# Load environment variables
load_dotenv(dotenv_path='.env')

# Initialize LLM with NVIDIA model
llm = NVIDIA(model="openai/gpt-oss-120b")

# Initialize embedding model
embedder = NVIDIAEmbedding(model="nvidia/nv-embedqa-e5-v5")

# Configure settings
Settings.llm = llm
Settings.embed_model = embedder

# Load documents from the data directory
documents = SimpleDirectoryReader("data").load_data()
print(f"Loaded {len(documents)} documents")

# Chunk documents to ensure they fit within embedding model limits
# NVIDIA embedqa-e5-v5 has a 512 token limit
text_splitter = TokenTextSplitter(
    chunk_size=200,          # Smaller chunks improve retrieval precision
    chunk_overlap=40         # Keep overlap so context is preserved
)

nodes = text_splitter.get_nodes_from_documents(documents)
print(f"Split into {len(nodes)} document chunks (max {text_splitter.chunk_size} tokens each)")

# Create vector index from chunked document nodes
index = VectorStoreIndex(nodes, embed_model=embedder)
query_engine = index.as_query_engine(
    similarity_top_k=7,
    response_mode="compact",
)

Loaded 7 documents
Split into 49 document chunks (max 280 tokens each)


## Example Queries

In [31]:
# Example 1: General knowledge from documents
query = "What are the main features of this project?"
response = query_engine.query(query)
print(f"Query: {query}")
print(f"\nResponse:\n{response}")

Query: What are the main features of this project?

Response:
The project focuses on building a clean, reliable API with several key capabilities:

- **Rich, structured error reporting** – errors are returned as detailed JSON objects that include a code, a clear message, and field‑specific information, making it easy for clients to understand what went wrong and how to fix it.  
- **Timezone‑aware timestamps** – all date‑time values are expressed in ISO 8601 format with an explicit UTC designator, ensuring consistent handling across different regions and systems.  
- **Thoughtful request design** – filters and parameters are placed where they belong (e.g., body vs. query string) instead of overloading query strings, leading to clearer and more maintainable endpoints.  
- **Secure OAuth 2.0 flow** – the implementation includes the full authorization‑code grant cycle with CSRF protection via state validation, proper handling of redirect URIs, and safe token exchange.  

Together, these f

In [32]:
# Example 2: Code implementation questions
query = "How do I implement a Retrieval Augmented Generation system with Python?"
response = query_engine.query(query)
print(f"Query: {query}")
print(f"\nResponse:\n{response}")

Query: How do I implement a Retrieval Augmented Generation system with Python?

Response:
Implementing a Retrieval‑Augmented Generation (RAG) pipeline in Python involves four main stages: preparing the data, creating searchable indexes, retrieving the most relevant pieces for a query, and feeding those pieces to a language model to generate a response. Below is a step‑by‑step guide with example code snippets that illustrate each stage.

---

## 1. Prepare the Documents (Chunking)

Large texts need to be broken into manageable pieces (chunks) so that they can be embedded and searched efficiently. You can chunk by size, by sentence, or by attaching metadata (e.g., source, section).

```python
def metadata_chunking(text, metadata_dict):
    """
    Split a document into chunks that carry both the raw text and associated metadata.
    """
    chunks = []
    for key, (text_part, meta_part) in metadata_dict.items():
        chunks.append({
            "text": text_part,
            "metadat

In [33]:
# Example 3: Dependency questions
query = "What dependencies are used for the RAG pipeline?"
response = query_engine.query(query)
print(f"Query: {query}")
print(f"\nResponse:\n{response}")

Query: What dependencies are used for the RAG pipeline?

Response:
I’m sorry, but the provided information doesn’t include details about the dependencies used for the RAG pipeline.


In [34]:
# Example 4: Installation and setup questions
query = "How do I install and configure this project?"
response = query_engine.query(query)
print(f"Query: {query}")
print(f"\nResponse:\n{response}")

Query: How do I install and configure this project?

Response:
The available information does not include any installation or configuration instructions for the project, so I’m unable to provide those details.
